In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
movies = pd.read_csv("../../datasets/raw/movielens/movies.csv")
ratings = pd.read_csv("../../datasets/raw/movielens/ratings.csv")

In [3]:
print("Movies Missing Values")
print(movies.isnull().sum())

print("\nRatings Missing Values")
print(ratings.isnull().sum())

Movies Missing Values
movieId    0
title      0
genres     0
dtype: int64

Ratings Missing Values
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [4]:
movies = movies.drop_duplicates()
ratings = ratings.drop_duplicates()

print("Duplicates Removed Successfully")

Duplicates Removed Successfully


In [5]:
print(movies.dtypes)
print(ratings.dtypes)

movieId    int64
title        str
genres       str
dtype: object
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object


In [6]:
movies["title"] = movies["title"].str.strip()

movies["title"] = movies["title"].str.replace(
    r"\s+",
    " ",
    regex=True
)

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)")

In [8]:
movies["clean_title"] = movies["title"].str.replace(
    r"\(\d{4}\)",
    "",
    regex=True
).str.strip()

In [9]:
movies["genres"] = movies["genres"].str.lower()

movies["genres"] = movies["genres"].str.replace("|", " ", regex=False)

In [10]:
movie_data = ratings.merge(
    movies,
    on="movieId"
)

movie_data.head()

,userId,movieId,rating,timestamp,title,genres,year,clean_title
0,1,1,4.0,964982703,Toy Story (1995),adventure animation children comedy fantasy,1995,Toy Story
1,1,3,4.0,964981247,Grumpier Old Men (1995),comedy romance,1995,Grumpier Old Men
2,1,6,4.0,964982224,Heat (1995),action crime thriller,1995,Heat
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),mystery thriller,1995,Seven (a.k.a. Se7en)
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",crime mystery thriller,1995,"Usual Suspects, The"


In [11]:
movie_data = movie_data[
    (movie_data["rating"] >= 0.5) &
    (movie_data["rating"] <= 5.0)
]

In [12]:
average_rating = movie_data.groupby(
    "movieId"
)["rating"].mean()

movie_data = movie_data.merge(
    average_rating.rename("average_rating"),
    on="movieId"
)

In [13]:
rating_count = movie_data.groupby(
    "movieId"
)["rating"].count()

movie_data = movie_data.merge(
    rating_count.rename("rating_count"),
    on="movieId"
)

In [14]:
movie_data = movie_data[
    movie_data["rating_count"] >= 5
]

In [15]:
movie_data["content"] = (
    movie_data["clean_title"] +
    " " +
    movie_data["genres"]
)

In [16]:
movie_data.head()

,userId,movieId,rating,timestamp,title,genres,year,clean_title,average_rating,rating_count,content
0,1,1,4.0,964982703,Toy Story (1995),adventure animation children comedy fantasy,1995,Toy Story,3.920930,215,Toy Story adventure animation children comedy ...
1,1,3,4.0,964981247,Grumpier Old Men (1995),comedy romance,1995,Grumpier Old Men,3.259615,52,Grumpier Old Men comedy romance
2,1,6,4.0,964982224,Heat (1995),action crime thriller,1995,Heat,3.946078,102,Heat action crime thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),mystery thriller,1995,Seven (a.k.a. Se7en),3.975369,203,Seven (a.k.a. Se7en) mystery thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",crime mystery thriller,1995,"Usual Suspects, The",4.237745,204,"Usual Suspects, The crime mystery thriller"


In [17]:
print(movie_data.shape)

(90274, 11)


In [18]:
movie_data.isnull().sum()

userId            0
movieId           0
rating            0
timestamp         0
title             0
genres            0
year              0
clean_title       0
average_rating    0
rating_count      0
content           0
dtype: int64

In [19]:
movie_data.to_csv(
    "../../datasets/processed/movie_data.csv",
    index=False
)

print("Processed Dataset Saved Successfully")

Processed Dataset Saved Successfully


In [20]:
print("="*60)
print("DATA PREPROCESSING COMPLETED")
print("="*60)

print("Movies :", movie_data["movieId"].nunique())
print("Users :", movie_data["userId"].nunique())
print("Ratings :", len(movie_data))

DATA PREPROCESSING COMPLETED
Movies : 3650
Users : 610
Ratings : 90274
